Please check the latest cell first (due to the better result and one structure code)

Part 1

In [5]:
import pandas as pd
import math
import re
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings("ignore") # ignoring all warnings that the code is faced

# Preprocessing the text by converting to lowercase, removing non-alphanumeric characters & handling numbers as a placeholder word
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)# Removing non-alphanumeric characters
    text = re.sub(r"\s+", " ", text).strip() # Replacing multiple or all white spaces with a single space
    words = re.findall(r'\b\w+\b', text)
    # Replacing any number (integer or with decimal) with the placeholder "numbers"
    for index, w in enumerate(words):
        if w.replace('.', '', 1).isdigit() or re.search(r'\d', w):
            words[index] = "numbers"
    return words

def load_data(file_path):
    return pd.read_csv(file_path)

def extract_text_column(df):
    col = df.columns[0]
    return df[col].astype(str).tolist() # Converting to a list of strings

# Generating n-grams (sequences of 'n' consecutive tokens) from our tokens
def custom_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

Part 2

In [6]:
class UnigramModel:
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self.model = Counter()
        self.vocab = set()
        self.total = 0

    def train(self, corpus):
        for sentence in corpus:
            tokens = preprocess(sentence)
            self.model.update(tokens)
            self.vocab.update(tokens)
        self.total = sum(self.model.values())

    # Calculating probability of a word in the model with additive smoothing
    def prob(self, word):
        return (self.model[word] + self.alpha) / (self.total + self.alpha * len(self.vocab))

    # Calculating perplexity of the model on a given corpus    
    def perplexity(self, corpus):
        log_prob = 0
        N = 0
        for sentence in corpus:
            tokens = preprocess(sentence)
            for word in tokens:
                log_prob += math.log(self.prob(word))
                N += 1
        return math.exp(-log_prob / N) if N > 0 else float('inf')

class KneserNeyModel:
    def __init__(self, n, discount=0.75):
        self.n = n
        self.discount = discount
        self.model = defaultdict(Counter)
        self.context_counts = defaultdict(int)
        self.continuation_counts = defaultdict(set)
        self.vocab = set()

    def train(self, corpus):
        for sentence in corpus:
            tokens = preprocess(sentence)
            tokens = ['<s>'] * (self.n-1) + tokens + ['</s>'] # Adding the start and end tokens
            for gram in custom_ngrams(tokens, self.n):
                context = gram[:-1]# Making the context be the first n-1 tokens
                word = gram[-1]
                self.model[context][word] += 1
                self.context_counts[context] += 1
                self.continuation_counts[word].add(context)
                self.vocab.add(word)

    def prob(self, context, word):
        context = tuple(context[-(self.n-1):]) # Taking the last n-1 tokens as context
        if context in self.model and word in self.model[context]:
            cont_prob = len(self.continuation_counts[word]) / len(self.context_counts)
            return max(self.model[context][word] - self.discount, 0) / self.context_counts[context] + \
                   self.discount * len(self.model[context]) / self.context_counts[context] * cont_prob
        else:
            return 1 / len(self.vocab) if len(self.vocab) > 0 else 0.0

    def perplexity(self, corpus):
        log_prob = 0
        N = 0
        for sentence in corpus:
            tokens = preprocess(sentence)
            tokens = ['<s>'] * (self.n-1) + tokens + ['</s>'] # Adding start and end tokens
            for gram in custom_ngrams(tokens, self.n):
                context = gram[:-1]
                word = gram[-1]
                prob = self.prob(context, word)
                if prob > 0:
                    log_prob += math.log(prob)
                    N += 1
        return math.exp(-log_prob / N) if N > 0 else float('inf')

Part 3

In [7]:
# Tuning the unigram model's alpha parameter to minimize perplexity on validation data
def tune_unigram_alpha(train_data, valid_data, alphas):
    best_alpha = alphas[0]
    best_perplexity = float('inf')
    for alpha in alphas:
        model = UnigramModel(alpha)
        model.train(train_data)
        perp = model.perplexity(valid_data)
        print(f"Alpha: {alpha}, Perplexity: {perp:.2f}")
        if perp < best_perplexity:
            best_perplexity = perp
            best_alpha = alpha
    return best_alpha, best_perplexity

Part 4

In [8]:
def main():
    train_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/train.csv")
    valid_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/val.csv")
    test_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/test.csv")
    train_text = extract_text_column(train_df)
    valid_text = extract_text_column(valid_df)
    test_text = extract_text_column(test_df)
    # Part 3
    best_alpha, _ = tune_unigram_alpha(train_text, valid_text, [0.01, 0.1, 0.5, 1.0, 1.5, 2.0, ])
    # Part 4
    # Training models & calculating perplexity on test data
    unigram_model = UnigramModel(alpha=best_alpha)
    unigram_model.train(train_text)
    unigram_perp = unigram_model.perplexity(test_text)

    bigram_model = KneserNeyModel(n=2)
    bigram_model.train(train_text)
    bigram_perp = bigram_model.perplexity(test_text)

    trigram_model = KneserNeyModel(n=3)
    trigram_model.train(train_text)
    trigram_perp = trigram_model.perplexity(test_text)
    # Print out the results
    print(f"Unigram Perplexity: {unigram_perp:.2f}")
    print(f"Bigram Perplexity: {bigram_perp:.2f}")
    print(f"Trigram Perplexity: {trigram_perp:.2f}")
    if min(unigram_perp, bigram_perp, trigram_perp) == unigram_perp:
        print("Unigram model provides the highest accuracy in predicting words.")
    elif min(unigram_perp, bigram_perp, trigram_perp) == bigram_perp:
        print("Bigram model provides the highest accuracy in predicting words.")
    else:
        print("Trigram model provides the highest accuracy in predicting words.")

if __name__ == "__main__":
    main()

Alpha: 0.01, Perplexity: 960.02
Alpha: 0.1, Perplexity: 921.55
Alpha: 0.5, Perplexity: 898.32
Alpha: 1.0, Perplexity: 892.04
Alpha: 1.5, Perplexity: 891.10
Alpha: 2.0, Perplexity: 892.49
Unigram Perplexity: 887.83
Bigram Perplexity: 154.07
Trigram Perplexity: 499.42
Bigram model provides the highest accuracy in predicting words.


New Version

In [10]:
# Part 1
import pandas as pd
import math
import re
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings("ignore") # ignoring all warnings that the code is faced

# Preprocessing the text by converting to lowercase, removing non-alphanumeric characters & handling numbers as a placeholder word
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)# Removing non-alphanumeric characters
    text = re.sub(r"\s+", " ", text).strip() # Replacing multiple or all white spaces with a single space
    words = re.findall(r'\b\w+\b', text)
    # Replacing any number (integer or with decimal) with the placeholder "numbers"
    for index, w in enumerate(words):
        if w.replace('.', '', 1).isdigit() or re.search(r'\d', w):
            words[index] = "numbers"
    return words

def load_data(file_path):
    return pd.read_csv(file_path)

def extract_text_column(df):
    col = df.columns[0]
    return df[col].astype(str).tolist() # Converting to a list of strings

# Generating n-grams (sequences of 'n' consecutive tokens) from our tokens
def custom_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    
# Part 2
class UnigramModel:
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self.model = Counter()
        self.vocab = set()
        self.total = 0

    def train(self, corpus):
        for sentence in corpus:
            tokens = preprocess(sentence)
            self.model.update(tokens)
            self.vocab.update(tokens)
        self.total = sum(self.model.values())
    
    # Calculating probability of a word in the model with additive smoothing
    def prob(self, word):
        return (self.model[word] + self.alpha) / (self.total + self.alpha * len(self.vocab))

    # Calculating perplexity of the model on a given corpus    
    def perplexity(self, corpus):
        log_prob = 0
        N = 0
        for sentence in corpus:
            tokens = preprocess(sentence)
            for word in tokens:
                prob = self.prob(word)
                if prob > 0:
                    log_prob += math.log(prob)
                    N += 1
        return math.exp(-log_prob / N) if N > 0 else float('inf')

class KneserNeyModel:
    def __init__(self, n, discount=0.75):
        self.n = n
        self.discount = discount
        self.model = defaultdict(Counter)
        self.context_counts = defaultdict(int)
        self.continuation_counts = defaultdict(set)
        self.vocab = set()
        self.total_contexts = 0
        self.lower_model = None # Lower-order model (for smoothing)
        # For all n > 2, recursively creating a lower-order model (n-1 grams)
        if n > 2:
            self.lower_model = KneserNeyModel(n-1, discount)

    # Training the model on the given corpus (list of sentences)
    def train(self, corpus):
        for sentence in corpus:
            tokens = preprocess(sentence)
            tokens = ['<s>'] * (self.n-1) + tokens + ['</s>'] # Adding start and end tokens
            for gram in custom_ngrams(tokens, self.n):
                context = gram[:-1]
                word = gram[-1]
                self.model[context][word] += 1
                self.context_counts[context] += 1
                self.continuation_counts[word].add(context)
                self.vocab.add(word)
        self.total_contexts = sum(len(self.model[ctx]) for ctx in self.model)
        if self.lower_model:
            self.lower_model.train(corpus) # Training the lower-order model for smoothing

    def prob(self, context, word):
        context = tuple(context[-(self.n-1):]) if len(context) >= self.n-1 else tuple(['<s>'] * (self.n-1 - len(context)) + list(context))
        if word not in self.vocab:
            return 1 / (len(self.vocab) + 1) if len(self.vocab) > 0 else 0.001
        lambda_weight = (self.discount * len(self.model[context]) / self.context_counts[context]) if self.context_counts[context] > 0 else 0   
        if self.context_counts[context] > 0:
            direct_prob = max(self.model[context][word] - self.discount, 0) / self.context_counts[context]
        else:
            direct_prob = 0        
        if self.n == 2:
            lower_prob = len(self.continuation_counts[word]) / self.total_contexts if self.total_contexts > 0 else 1 / (len(self.vocab) + 1)
        else:
            lower_prob = self.lower_model.prob(context[1:], word) if self.lower_model else 1 / (len(self.vocab) + 1)
        return direct_prob + lambda_weight * lower_prob

    def perplexity(self, corpus):
        log_prob = 0
        N = 0
        for sentence in corpus:
            tokens = preprocess(sentence)
            tokens = ['<s>'] * (self.n-1) + tokens + ['</s>']
            for gram in custom_ngrams(tokens, self.n):
                context = gram[:-1]
                word = gram[-1]
                prob = self.prob(context, word)
                if prob > 0:
                    log_prob += math.log(prob)
                    N += 1
        return math.exp(-log_prob / N) if N > 0 else float('inf')

# Part 3
# Tuning the unigram model's alpha to minimize perplexity (grid search)
def tune_unigram_alpha(train_data, valid_data, alphas):
    best_alpha = alphas[0]
    best_perplexity = float('inf')
    for alpha in alphas:
        model = UnigramModel(alpha)
        model.train(train_data)
        perp = model.perplexity(valid_data)
        print(f"Unigram Alpha: {alpha}, Perplexity: {perp:.2f}")
        if perp < best_perplexity:
            best_perplexity = perp
            best_alpha = alpha
    return best_alpha, best_perplexity

# Tuning Kneser-Ney model's discount parameter to minimize perplexity
def tune_kneser_ney(train_data, valid_data, n, discounts):
    best_discount = discounts[0]
    best_perplexity = float('inf')
    for discount in discounts:
        model = KneserNeyModel(n=n, discount=discount)
        model.train(train_data)
        perp = model.perplexity(valid_data)
        print(f"N={n}, Discount: {discount}, Perplexity: {perp:.2f}")
        # Updating the best perplexity
        if perp < best_perplexity:
            best_perplexity = perp
            best_discount = discount
    return best_discount, best_perplexity

# Part 4
def main():
    train_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/train.csv")
    valid_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/val.csv")
    test_df = load_data("E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/test.csv")
    train_text = extract_text_column(train_df)
    valid_text = extract_text_column(valid_df)
    test_text = extract_text_column(test_df)
    # Part 3
    # Tuning the unigram model's alpha parameter and the Kneser-Ney model's discount parameters
    best_alpha, _ = tune_unigram_alpha(train_text, valid_text, [0.01, 0.1, 0.5, 1.0, 1.5, 2.0])
    best_bigram_discount, _ = tune_kneser_ney(train_text, valid_text, n=2, discounts=[0.1, 0.3, 0.5, 0.75, 0.9])
    best_trigram_discount, _ = tune_kneser_ney(train_text, valid_text, n=3, discounts=[0.1, 0.3, 0.5, 0.75, 0.9])

    # Part 4
    # Training models & calculating perplexity on test data
    unigram_model = UnigramModel(alpha=best_alpha)
    unigram_model.train(train_text)
    unigram_perp = unigram_model.perplexity(test_text)

    bigram_model = KneserNeyModel(n=2, discount=best_bigram_discount)
    bigram_model.train(train_text)
    bigram_perp = bigram_model.perplexity(test_text)

    trigram_model = KneserNeyModel(n=3, discount=best_trigram_discount)
    trigram_model.train(train_text)
    trigram_perp = trigram_model.perplexity(test_text)

    print(f"\nFinal Results on Test Set:")
    print(f"Unigram Perplexity: {unigram_perp:.2f}")
    print(f"Bigram Perplexity: {bigram_perp:.2f}")
    print(f"Trigram Perplexity: {trigram_perp:.2f}")
    min_perp = min(unigram_perp, bigram_perp, trigram_perp)
    if min_perp == unigram_perp:
        print("Unigram model provides the highest accuracy in predicting words.")
    elif min_perp == bigram_perp:
        print("Bigram model provides the highest accuracy in predicting words.")
    else:
        print("Trigram model provides the highest accuracy in predicting words.")

if __name__ == "__main__":
    main()

Unigram Alpha: 0.01, Perplexity: 960.02
Unigram Alpha: 0.1, Perplexity: 921.55
Unigram Alpha: 0.5, Perplexity: 898.32
Unigram Alpha: 1.0, Perplexity: 892.04
Unigram Alpha: 1.5, Perplexity: 891.10
Unigram Alpha: 2.0, Perplexity: 892.49
N=2, Discount: 0.1, Perplexity: 194.45
N=2, Discount: 0.3, Perplexity: 165.27
N=2, Discount: 0.5, Perplexity: 155.96
N=2, Discount: 0.75, Perplexity: 153.21
N=2, Discount: 0.9, Perplexity: 156.70
N=3, Discount: 0.1, Perplexity: 180.18
N=3, Discount: 0.3, Perplexity: 112.45
N=3, Discount: 0.5, Perplexity: 93.10
N=3, Discount: 0.75, Perplexity: 84.55
N=3, Discount: 0.9, Perplexity: 86.17

Final Results on Test Set:
Unigram Perplexity: 887.83
Bigram Perplexity: 152.98
Trigram Perplexity: 83.35
Trigram model provides the highest accuracy in predicting words.
